# 🏢 HVAC-RL 分阶段Pipeline (推荐)

## ✨ 特点
- ✅ **分阶段运行**: 每个阶段独立运行，可随时暂停
- ✅ **实时Drive备份**: 每个episode完成立即保存，断开后可续
- ✅ **自动恢复**: 断点续传，不浪费GPU
- ✅ **清晰路径**: 每个阶段显示输入/输出文件位置
- ✅ **快速测试**: 10 episodes版本，2-3小时完成

## 📂 数据流向图
```
Stage 1: PPO训练
  输出 → pipeline_output/01_ppo_training/ppo_trajectory.json
         ↓
Stage 2: Few-shot选择
  输入 ← ppo_trajectory.json
  输出 → pipeline_output/02_few_shot_samples/few_shot_examples_structured.json
         ↓
Stage 3: LLM Rollout (10 episodes × 200 steps = 2000 steps)
  输入 ← few_shot_examples_structured.json
  输出 → pipeline_output/03_llm_rollout/llm_rollout.json
  ☁️  每个episode完成后立即备份到Drive！
         ↓
Stage 4: Fine-tuning (包含自我蒸馏)
  输入 ← llm_rollout.json
  输出 → pipeline_output/04_finetuning/final_model/
         ↓
Stage 5: 评估
  输入 ← final_model/
  输出 → pipeline_output/05_evaluation/comparison_plot.png
```

---
## 🔧 Part 0: 环境设置 (只需运行一次)

In [ ]:
# 0.1 检查GPU
import torch
import os

print("=" * 60)
print("GPU信息")
print("=" * 60)
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU型号: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU显存: {gpu_mem:.1f} GB")
    
    if gpu_mem < 30:
        print("⚠️  警告: 显存小于30GB，建议使用A100 (40GB)")
    else:
        print("✅ GPU配置充足！")
else:
    print("❌ 未检测到GPU！")
    print("   Runtime > Change runtime type > Hardware accelerator > GPU")

In [ ]:
# 0.2 挂载Google Drive (用于保存结果和实时备份)
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive已挂载")

In [ ]:
# 0.3 克隆项目 (如果已存在会拉取最新更新)
PROJECT_DIR = "/content/HVAC-RL"

if os.path.exists(PROJECT_DIR):
    print("项目已存在，拉取最新更新...")
    !cd {PROJECT_DIR} && git fetch && git checkout claude/setup-project-verification-6y4Ow && git pull
else:
    print("克隆项目...")
    !git clone -b claude/setup-project-verification-6y4Ow https://github.com/Mo119m/HAVC-control-with-reinforcement-learning-update.git {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f"\n✅ 当前目录: {os.getcwd()}")

In [ ]:
# 0.4 安装依赖
print("安装依赖包...")
!pip install -q torch transformers accelerate peft
!pip install -q stable-baselines3 sb3-contrib gymnasium
!pip install -q numpy pandas scikit-learn scipy matplotlib seaborn
!pip install -q pvlib cvxpy tqdm

print("\n✅ 所有依赖已安装！")

---
## ⚙️ Part 1: 配置选择

In [ ]:
# 1.1 选择配置模式
import json

print("选择配置模式:\n")
print("1️⃣  快速测试 (10 episodes, ~2-3小时)")
print("    - LLM Rollout: 10 episodes × 200 steps = 2,000 steps")
print("    - 适合: 验证功能、快速迭代、调试\n")

print("2️⃣  完整训练 (50 episodes, ~3-5小时)")
print("    - LLM Rollout: 50 episodes × 200 steps = 10,000 steps")
print("    - 适合: 最终实验、论文数据\n")

# 默认使用快速测试配置
USE_QUICK_TEST = True  # 改为 False 使用完整配置

if USE_QUICK_TEST:
    print("✅ 使用: 快速测试配置 (10 episodes)")
    CONFIG_FILE = "config_test_10ep.json"
    EPISODES = 10
else:
    print("✅ 使用: 完整训练配置 (50 episodes)")
    CONFIG_FILE = "config_enhanced.json"
    EPISODES = 50

In [ ]:
# 1.2 创建配置并启用实时Drive备份 ⭐ 重要！
print("=" * 60)
print("创建配置并启用实时Drive备份")
print("=" * 60)

# 读取增强配置
with open('config_enhanced.json', 'r') as f:
    config = json.load(f)

# 修改episodes数量
if USE_QUICK_TEST:
    config['llm_rollout_episodes'] = 10
    config['_comment_llm'] = "LLM rollout configuration - 10 episodes for quick testing"
    print("\n📝 配置: 快速测试模式 (10 episodes)")
else:
    config['llm_rollout_episodes'] = 50
    config['_comment_llm'] = "LLM rollout configuration - 50 episodes for full training"
    print("\n📝 配置: 完整训练模式 (50 episodes)")

# ⭐ 启用实时Drive备份 (防止断开丢失数据)
config['drive_backup_path'] = '/content/drive/MyDrive/HVAC-RL-Backup'
config['resume_from_backup'] = True

# 保存配置
with open(CONFIG_FILE, 'w') as f:
    json.dump(config, f, indent=2)

print(f"\n✅ 配置已创建: {CONFIG_FILE}")
print(f"   Episodes: {config['llm_rollout_episodes']}")
print(f"   Steps per episode: {config['llm_rollout_max_steps']}")
print(f"   Total steps: {config['llm_rollout_episodes'] * config['llm_rollout_max_steps']:,}")
print(f"\n☁️  实时Drive备份已启用！")
print(f"   备份位置: {config['drive_backup_path']}")
print(f"   ✨ 每个episode完成后立即保存到Drive")
print(f"   ✨ 断开后可以从上次的episode继续")
print(f"   ✨ 零数据丢失，不浪费GPU！")

---
## 📂 Part 2: 文件路径检查工具

In [ ]:
# 2.1 显示所有阶段的文件路径和Drive备份状态
from pathlib import Path
from datetime import datetime

def check_pipeline_files():
    """检查并显示所有pipeline文件的状态"""
    
    base_dir = Path("./pipeline_output")
    drive_backup = Path("/content/drive/MyDrive/HVAC-RL-Backup")
    
    print("=" * 80)
    print("Pipeline 文件状态检查")
    print("=" * 80)
    print(f"检查时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"本地目录: {base_dir.absolute()}")
    print(f"Drive备份: {drive_backup}\n")
    
    stages = [
        {
            "name": "Stage 1: PPO训练",
            "dir": "01_ppo_training",
            "key_file": "ppo_trajectory.json",
        },
        {
            "name": "Stage 2: Few-shot选择",
            "dir": "02_few_shot_samples",
            "key_file": "few_shot_examples_structured.json",
        },
        {
            "name": f"Stage 3: LLM Rollout ({EPISODES} episodes)",
            "dir": "03_llm_rollout",
            "key_file": "llm_rollout.json",
        },
        {
            "name": "Stage 4: Fine-tuning",
            "dir": "04_finetuning",
            "key_file": "final_model",
        },
        {
            "name": "Stage 5: 评估",
            "dir": "05_evaluation",
            "key_file": "comparison_plot.png",
        },
    ]
    
    for stage in stages:
        print(f"\n{'─' * 80}")
        print(f"{stage['name']}")
        print(f"{'─' * 80}")
        
        # 检查本地文件
        local_path = base_dir / stage['dir'] / stage['key_file']
        if local_path.exists():
            if local_path.is_dir():
                print(f"📁 本地: ✅ {stage['key_file']} (目录存在)")
            else:
                size_mb = local_path.stat().st_size / 1024 / 1024
                mtime = datetime.fromtimestamp(local_path.stat().st_mtime)
                print(f"📁 本地: ✅ {stage['key_file']} ({size_mb:.2f} MB)")
                print(f"   修改时间: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        else:
            print(f"📁 本地: ⏳ {stage['key_file']} (未生成)")
        
        # 检查Drive备份
        drive_path = drive_backup / stage['dir'] / stage['key_file']
        if drive_path.exists():
            if drive_path.is_dir():
                print(f"☁️  Drive: ✅ 已备份")
            else:
                size_mb = drive_path.stat().st_size / 1024 / 1024
                mtime = datetime.fromtimestamp(drive_path.stat().st_mtime)
                print(f"☁️  Drive: ✅ 已备份 ({size_mb:.2f} MB)")
                print(f"   备份时间: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        else:
            print(f"☁️  Drive: ⏳ 未备份")
        
        # Stage 3 特殊检查：显示episode备份
        if stage['dir'] == '03_llm_rollout' and drive_path.exists():
            episode_backups = list((drive_backup / stage['dir']).glob("llm_rollout_ep*.json"))
            if episode_backups:
                print(f"   📊 Episode备份: {len(episode_backups)} 个")
    
    print(f"\n{'=' * 80}")

# 运行检查
check_pipeline_files()

---
## 🚀 Part 3: 分阶段运行

### 💡 使用说明
- 按顺序运行每个Stage
- Stage 3 每完成1个episode会立即保存到Drive
- 如果Colab断开，运行"快速恢复"(Part 7.1)，然后继续运行Stage 3
- 随时运行"文件路径检查"(Part 2)查看进度

In [ ]:
# Stage 1: PPO训练
print("=" * 80)
print("Stage 1: PPO训练")
print("=" * 80)
print("输入: 无 (从零开始训练)")
print("输出: pipeline_output/01_ppo_training/ppo_trajectory.json")
print("预计时间: 1-2小时 (500,000 steps)\n")

!python core_modules/main_pipeline.py --config {CONFIG_FILE} --stage ppo

print("\n✅ Stage 1 完成！")
print("下一步: 运行 Stage 2 (Few-shot选择)")

In [ ]:
# Stage 2: Few-shot选择
print("=" * 80)
print("Stage 2: Few-shot示例选择")
print("=" * 80)
print("输入: pipeline_output/01_ppo_training/ppo_trajectory.json")
print("输出: pipeline_output/02_few_shot_samples/few_shot_examples_structured.json")
print("预计时间: 5-10分钟\n")

!python core_modules/main_pipeline.py --config {CONFIG_FILE} --stage select

print("\n✅ Stage 2 完成！")
print("下一步: 运行 Stage 3 (LLM Rollout)")

In [ ]:
# Stage 3: LLM Rollout (多episode + 实时Drive备份)
print("=" * 80)
print(f"Stage 3: LLM Rollout ({EPISODES} episodes)")
print("=" * 80)
print("输入: pipeline_output/02_few_shot_samples/few_shot_examples_structured.json")
print("输出: pipeline_output/03_llm_rollout/llm_rollout.json")
print(f"数据量: {EPISODES} episodes × 200 steps = {EPISODES * 200:,} steps")
print(f"预计时间: {10 if USE_QUICK_TEST else 30}-{20 if USE_QUICK_TEST else 60}分钟")
print(f"\n☁️  实时备份已启用: 每个episode完成后立即保存到Drive")
print(f"   断开后可以从上次的episode继续！\n")

!python core_modules/main_pipeline.py --config {CONFIG_FILE} --stage rollout

print("\n✅ Stage 3 完成！")
print("下一步: 运行 Stage 4 (Fine-tuning)")

In [ ]:
# Stage 4: Fine-tuning (包含自我蒸馏)
print("=" * 80)
print("Stage 4: Fine-tuning with Self-Distillation")
print("=" * 80)
print("输入: pipeline_output/03_llm_rollout/llm_rollout.json")
print("输出: pipeline_output/04_finetuning/final_model/")
print("预计时间: 1-2小时\n")

!python core_modules/main_pipeline.py --config {CONFIG_FILE} --stage finetune

print("\n✅ Stage 4 完成！")
print("下一步: 运行 Stage 5 (评估)")

In [ ]:
# Stage 5: 评估
print("=" * 80)
print("Stage 5: 评估和对比")
print("=" * 80)
print("输入: pipeline_output/04_finetuning/final_model/")
print("输出: pipeline_output/05_evaluation/comparison_plot.png")
print("预计时间: 10-20分钟\n")

!python core_modules/main_pipeline.py --config {CONFIG_FILE} --stage eval

print("\n✅ Stage 5 完成！")
print("🎉 所有阶段已完成！")

---
## 🎯 Part 4: 一键运行所有阶段 (可选)

如果你想一次性运行所有阶段，运行下面这个cell：

In [ ]:
# 一键运行所有阶段
import time

start_time = time.time()

print("=" * 80)
print(f"开始运行完整Pipeline ({EPISODES} episodes)")
print("=" * 80)
print(f"配置: {CONFIG_FILE}")
print(f"LLM Rollout: {EPISODES} episodes × 200 steps = {EPISODES * 200:,} steps")
print(f"预计时间: {'2-3小时' if USE_QUICK_TEST else '3-5小时'}")
print(f"☁️  实时Drive备份已启用\n")

!python core_modules/main_pipeline.py --config {CONFIG_FILE} --stage all

elapsed_hours = (time.time() - start_time) / 3600
elapsed_mins = (time.time() - start_time) / 60

print("\n" + "=" * 80)
print("🎉 Pipeline完成！")
print("=" * 80)
if elapsed_hours >= 1:
    print(f"总耗时: {elapsed_hours:.2f} 小时")
else:
    print(f"总耗时: {elapsed_mins:.1f} 分钟")
print("\n下一步: 运行 Part 5 查看结果")

---
## 📊 Part 5: 查看结果

In [ ]:
# 5.1 再次检查所有文件和Drive备份
check_pipeline_files()

In [ ]:
# 5.2 显示最终对比图
import matplotlib.pyplot as plt
from PIL import Image

comparison_plot = "./pipeline_output/05_evaluation/comparison_plot.png"

if os.path.exists(comparison_plot):
    img = Image.open(comparison_plot)
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('PPO vs LLM vs Fine-tuned LLM 对比', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("❌ 对比图未找到")
    print("   请确保 Stage 5 (评估) 已完成")

In [ ]:
# 5.3 计算详细统计
import json
import numpy as np

def load_and_analyze_trajectory(path, name):
    """加载并分析轨迹数据"""
    if not os.path.exists(path):
        print(f"❌ {name}: 文件不存在 ({path})")
        return None
    
    with open(path, 'r') as f:
        traj = json.load(f)
    
    if isinstance(traj, list) and len(traj) > 0:
        rewards = [step.get('reward', 0) for step in traj]
        
        print(f"\n{'─' * 60}")
        print(f"{name}")
        print(f"{'─' * 60}")
        print(f"总步数: {len(rewards):,}")
        print(f"平均reward: {np.mean(rewards):.4f}")
        print(f"标准差: {np.std(rewards):.4f}")
        print(f"最小值: {np.min(rewards):.4f}")
        print(f"最大值: {np.max(rewards):.4f}")
        print(f"总reward: {np.sum(rewards):.2f}")
        print(f"中位数: {np.median(rewards):.4f}")
        
        # 检查是否有episode信息
        if 'episode' in traj[0]:
            episodes = set(step.get('episode', 0) for step in traj)
            print(f"Episodes数量: {len(episodes)}")
        
        return rewards
    else:
        print(f"❌ {name}: 数据格式错误")
        return None

print("=" * 80)
print("性能统计分析")
print("=" * 80)

ppo_rewards = load_and_analyze_trajectory(
    './pipeline_output/01_ppo_training/ppo_trajectory.json',
    '📊 PPO Expert (Baseline)'
)

llm_rewards = load_and_analyze_trajectory(
    './pipeline_output/03_llm_rollout/llm_rollout.json',
    f'🤖 LLM Before Fine-tuning ({EPISODES} episodes)'
)

ft_rewards = load_and_analyze_trajectory(
    './pipeline_output/05_evaluation/finetuned_rollout.json',
    '🎓 LLM After Fine-tuning'
)

# 计算改进幅度
if llm_rewards and ft_rewards:
    improvement = (np.mean(ft_rewards) - np.mean(llm_rewards)) / abs(np.mean(llm_rewards)) * 100
    print(f"\n{'─' * 60}")
    print("📈 Fine-tuning改进")
    print(f"{'─' * 60}")
    print(f"平均reward提升: {improvement:+.2f}%")
    
    if improvement > 0:
        print("✅ Fine-tuning成功！模型性能提升")
    else:
        print("⚠️  Fine-tuning后性能下降，可能需要:")
        print("   - 增加rollout episodes数量")
        print("   - 调整learning rate")
        print("   - 增加训练epochs")

if ppo_rewards and ft_rewards:
    gap = (np.mean(ft_rewards) - np.mean(ppo_rewards)) / abs(np.mean(ppo_rewards)) * 100
    print(f"\n与PPO Expert差距: {gap:+.2f}%")
    if gap > -10:
        print("✅ 接近专家水平！")
    elif gap > -30:
        print("⚠️  仍有改进空间")
    else:
        print("❌ 与专家差距较大，建议增加训练数据")

print("\n" + "=" * 80)

---
## 💾 Part 6: 额外保存结果副本

In [ ]:
# 6.1 保存所有结果副本到Drive (带时间戳)
import shutil
from datetime import datetime

# 创建带时间戳的目录
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
config_tag = f"{EPISODES}ep" if USE_QUICK_TEST else "50ep_full"
save_dir = f"/content/drive/MyDrive/HVAC_RL_Results/run_{config_tag}_{timestamp}"
os.makedirs(save_dir, exist_ok=True)

print("=" * 80)
print("保存结果副本到Google Drive")
print("=" * 80)
print(f"目标目录: {save_dir}\n")

# 复制pipeline输出
if os.path.exists("./pipeline_output"):
    print("复制pipeline_output...")
    shutil.copytree(
        "./pipeline_output",
        f"{save_dir}/pipeline_output",
        dirs_exist_ok=True
    )
    print("✅ Pipeline输出已保存")

# 保存配置
print("\n保存配置文件...")
shutil.copy(CONFIG_FILE, f"{save_dir}/{CONFIG_FILE}")
print(f"✅ 配置已保存: {CONFIG_FILE}")

# 创建运行摘要
summary = {
    "timestamp": timestamp,
    "config": CONFIG_FILE,
    "episodes": EPISODES,
    "total_steps": EPISODES * 200,
    "mode": "quick_test" if USE_QUICK_TEST else "full_training",
}

with open(f"{save_dir}/run_summary.json", 'w') as f:
    json.dump(summary, f, indent=2)
print("✅ 运行摘要已保存")

print("\n" + "=" * 80)
print("✅ 结果副本已保存到Google Drive!")
print("=" * 80)
print(f"📂 位置: {save_dir}")
print("\n💡 注意: 这是额外副本，实时备份已在 /HVAC-RL-Backup/ 中")

---
## 🔧 Part 7: 工具和故障排除

In [ ]:
# 7.1 快速恢复 (Colab断开后使用) ⭐ 重要！
print("=" * 60)
print("快速恢复 - Colab断开后从这里开始")
print("=" * 60)

# 重新挂载Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 切换到项目目录
PROJECT_DIR = "/content/HVAC-RL"
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
    print(f"✅ 已切换到: {os.getcwd()}\n")
    
    # 检查Drive备份并恢复到本地
    import shutil
    drive_backup = Path("/content/drive/MyDrive/HVAC-RL-Backup")
    
    if drive_backup.exists():
        print("☁️  找到Drive备份，恢复到本地...\n")
        
        # 恢复各个阶段的数据
        for stage_dir in ["01_ppo_training", "02_few_shot_samples", "03_llm_rollout", "04_finetuning", "05_evaluation"]:
            src = drive_backup / stage_dir
            dst = Path("./pipeline_output") / stage_dir
            
            if src.exists():
                os.makedirs(dst.parent, exist_ok=True)
                try:
                    shutil.copytree(str(src), str(dst), dirs_exist_ok=True)
                    print(f"   ✅ 恢复: {stage_dir}")
                except Exception as e:
                    print(f"   ⚠️  {stage_dir}: {e}")
        
        print("\n✅ 数据恢复完成！")
    else:
        print("⚠️  未找到Drive备份")
    
    # 检查pipeline状态
    print("\n" + "=" * 60)
    check_pipeline_files()
    
    print("\n💡 下一步: 重新运行未完成的Stage (会自动跳过已完成的episodes)")
else:
    print("❌ 项目不存在，请重新运行 Part 0 (环境设置)")

In [ ]:
# 7.2 监控GPU
!nvidia-smi

In [ ]:
# 7.3 检查磁盘空间
!df -h | grep -E 'Filesystem|/content'

In [ ]:
# 7.4 清理缓存
import torch
import gc

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ GPU缓存已清理")

gc.collect()
print("✅ 内存已清理")

---
## 📝 使用说明

### 🎯 推荐工作流程

**首次运行:**
1. Part 0: 环境设置 (4个cells)
2. Part 1: 配置选择 (2个cells) - 会自动启用Drive备份
3. Part 2: 检查文件状态
4. Part 3: 依次运行 Stage 1-5
5. Part 5: 查看结果

**Colab断开后恢复:**
1. Part 7.1: 快速恢复 ⭐ 一键从Drive恢复所有数据
2. Part 2: 检查已完成的Stages
3. Part 3: 重新运行对应Stage (会自动跳过已完成的episodes)

### ⏱️ 预计时间

**快速测试 (10 episodes):**
- Stage 1: 1-2小时
- Stage 2: 5-10分钟
- Stage 3: 10-20分钟 (每个episode ~1-2分钟)
- Stage 4: 30-60分钟
- Stage 5: 10-20分钟
- **总计**: ~2-3小时

**完整训练 (50 episodes):**
- Stage 1: 1-2小时
- Stage 2: 5-10分钟
- Stage 3: 30-60分钟 (每个episode ~1-2分钟)
- Stage 4: 1-2小时
- Stage 5: 10-20分钟
- **总计**: ~3-5小时

### 💡 重要提示

1. **实时备份**: Stage 3每个episode完成后立即保存到Drive
2. **断点续传**: 断开后运行Part 7.1恢复，然后继续Stage 3
3. **零数据丢失**: 所有数据都有Drive备份
4. **不浪费GPU**: 已完成的episodes会自动跳过

### 🐛 常见问题

**Q: Colab断开了怎么办？**
- 运行 Part 7.1 快速恢复
- 运行 Part 2 查看进度
- 重新运行对应Stage（会自动续传）

**Q: 如何查看已完成的episodes？**
- 运行 Part 2 文件检查
- 查看"Episode备份"数量

**Q: Stage 3运行时可以看到实时备份吗？**
- 是的！每个episode完成后会显示:
  - `💾 Saved to local`
  - `☁️ Backed up to Drive`
  - `☁️ Episode backup`

**Q: Fine-tuning效果不好？**
- 增加 episodes (改Part 1.1中的USE_QUICK_TEST)
- 10 episodes → 2,000 steps
- 50 episodes → 10,000 steps (推荐)